# PHASE 4: BỘ QUYẾT ĐỊNH NETWORK
## Decision Engine với Adaptive Threshold

In [1]:
import os
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, roc_curve, auc
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import train_test_split
import pickle
import time
import json
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

os.makedirs('Phase4_Network_Models', exist_ok=True)
os.makedirs('Phase4_Network_Data', exist_ok=True)

print("=" * 60)
print("PHASE 4: BỘ QUYẾT ĐỊNH NETWORK")
print("=" * 60)

PHASE 4: BỘ QUYẾT ĐỊNH NETWORK


## BƯỚC 1: LOAD MODEL

In [2]:
print("\nBƯỚC 1: LOAD MODEL")
print("-" * 40)

with open('Phase2_Network_Models/best_model_info.pkl', 'rb') as f:
    model_info = pickle.load(f)

best_model_name = model_info['best_model_name']
model_filename = model_info['model_filename']

with open(f'Phase2_Network_Models/{model_filename}', 'rb') as f:
    best_model = pickle.load(f)

print(f"Model: {best_model_name}")
print(f"F1-Score: {model_info['test_f1_score']:.4f}")

X_test = pd.read_csv('Phase1_Network_Data/X_test_processed.csv')
y_test = pd.read_csv('Phase1_Network_Data/y_test_processed.csv').iloc[:, 0]


BƯỚC 1: LOAD MODEL
----------------------------------------
Model: XGBoost
F1-Score: 0.9028


## BƯỚC 2: TÌM OPTIMAL THRESHOLD

In [3]:
print("\nBƯỚC 2: TÌM OPTIMAL THRESHOLD")
print("-" * 40)

y_proba = best_model.predict_proba(X_test)[:, 1]
fpr, tpr, thresholds = roc_curve(y_test, y_proba)
roc_auc = auc(fpr, tpr)

optimal_idx = np.argmax(tpr >= 0.9)
optimal_threshold = thresholds[optimal_idx]

print(f"ROC AUC: {roc_auc:.4f}")
print(f"Optimal threshold: {optimal_threshold:.4f}")
print(f"TPR: {tpr[optimal_idx]:.4f}, FPR: {fpr[optimal_idx]:.4f}")


BƯỚC 2: TÌM OPTIMAL THRESHOLD
----------------------------------------
ROC AUC: 0.9856
Optimal threshold: 0.1064
TPR: 0.9000, FPR: 0.0504


## BƯỚC 3: CALIBRATE MODEL

In [4]:
print("\nBƯỚC 3: CALIBRATE MODEL")
print("-" * 40)

X_cal, X_eval, y_cal, y_eval = train_test_split(
    X_test, y_test, test_size=0.8, random_state=42, stratify=y_test
)

calibrated_model = CalibratedClassifierCV(best_model, method='isotonic', cv=3)
calibrated_model.fit(X_cal, y_cal)

print(f"Calibration: {X_cal.shape}, Evaluation: {X_eval.shape}")


BƯỚC 3: CALIBRATE MODEL
----------------------------------------
Calibration: (35068, 42), Evaluation: (140273, 42)


## BƯỚC 4: DECISION ENGINE

In [5]:
print("\nBƯỚC 4: DECISION ENGINE")
print("-" * 40)

class NetworkDecisionEngine:
    def __init__(self, model, optimal_threshold=0.5, attack_bias_factor=1.3):
        self.model = model
        self.optimal_threshold = optimal_threshold
        self.attack_bias_factor = attack_bias_factor
        self.detection_log = []
    
    def predict(self, sample):
        start_time = time.time()
        
        if len(sample.shape) == 1:
            sample = sample.reshape(1, -1)
        elif isinstance(sample, pd.DataFrame):
            sample = sample.values
        
        probabilities = self.model.predict_proba(sample)[0]
        attack_prob = probabilities[1]
        
        adaptive_threshold = self.optimal_threshold / self.attack_bias_factor
        
        if attack_prob >= adaptive_threshold:
            predicted_class = 1
            confidence = attack_prob
            attack_name = 'Attack'
            threat_level = 'CRITICAL'
        else:
            predicted_class = 0
            confidence = 1.0 - attack_prob
            attack_name = 'Normal'
            threat_level = 'NORMAL'
        
        processing_time = (time.time() - start_time) * 1000
        
        result = {
            'prediction': predicted_class,
            'attack_name': attack_name,
            'confidence': confidence,
            'attack_probability': attack_prob,
            'threat_level': threat_level,
            'processing_time_ms': processing_time
        }
        
        self.detection_log.append(result)
        return result

network_engine = NetworkDecisionEngine(
    model=calibrated_model,
    optimal_threshold=optimal_threshold,
    attack_bias_factor=1.3
)

print("Engine created")


BƯỚC 4: DECISION ENGINE
----------------------------------------
Engine created


## BƯỚC 5: ĐÁNH GIÁ

In [6]:
print("\nBƯỚC 5: ĐÁNH GIÁ")
print("-" * 40)

# Use batch prediction instead of loop (much faster)
all_predictions = network_engine.model.predict(X_eval)

accuracy = accuracy_score(y_eval, all_predictions)
precision = precision_score(y_eval, all_predictions, average='weighted', zero_division=0)
recall = recall_score(y_eval, all_predictions, average='weighted', zero_division=0)
f1 = f1_score(y_eval, all_predictions, average='weighted', zero_division=0)

print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-Score: {f1:.4f}")


BƯỚC 5: ĐÁNH GIÁ
----------------------------------------
Accuracy: 0.9526
Precision: 0.9525
Recall: 0.9526
F1-Score: 0.9523


## BƯỚC 6: LƯU KẾT QUẢ

In [7]:
print("\nBƯỚC 6: LƯU KẾT QUẢ")
print("-" * 40)

with open('Phase4_Network_Models/network_decision_engine.pkl', 'wb') as f:
    pickle.dump(network_engine, f)

final_metrics = {
    'accuracy': accuracy,
    'precision': precision,
    'recall': recall,
    'f1_score': f1,
    'model_type': best_model_name
}

with open('Phase4_Network_Data/final_metrics.json', 'w') as f:
    json.dump(final_metrics, f, indent=2)

print("✅ Đã lưu engine và metrics")


BƯỚC 6: LƯU KẾT QUẢ
----------------------------------------
✅ Đã lưu engine và metrics


## BƯỚC 7: TEST TRÊN RAW DATASET

In [8]:
print("\nBƯỚC 7: TEST TRÊN RAW DATASET")
print("-" * 40)

# Load raw test data
test_raw = pd.read_csv('Network/UNSW_NB15_testing-set.csv')
print(f"Raw test data: {test_raw.shape}")

# Drop unnecessary columns (same as Phase 1)
cols_to_drop = ['id', 'attack_cat', 'label']
cols_to_drop = [col for col in cols_to_drop if col in test_raw.columns]

# Separate features and target
X_test_raw = test_raw.drop(cols_to_drop, axis=1)
y_test_raw = test_raw['label']

# Load preprocessing artifacts
with open('Phase1_Network_Models/network_feature_encoders.pkl', 'rb') as f:
    feature_encoders = pickle.load(f)
with open('Phase1_Network_Models/network_scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)

# Apply encoding to categorical columns
X_test_encoded = X_test_raw.copy()
for col, encoder in feature_encoders.items():
    if col in X_test_encoded.columns:
        test_values = X_test_encoded[col].astype(str).values
        class_to_idx = {cls: idx for idx, cls in enumerate(encoder.classes_)}
        X_test_encoded[col] = np.array([class_to_idx.get(val, 0) for val in test_values])

# Apply scaling (CRITICAL FIX)
X_test_scaled = scaler.transform(X_test_encoded)

print(f"Features after encoding and scaling: {X_test_scaled.shape}")

# Predict with decision engine (batch prediction for speed)
all_predictions_raw = network_engine.model.predict(X_test_scaled)

# Evaluate
accuracy_raw = accuracy_score(y_test_raw, all_predictions_raw)
precision_raw = precision_score(y_test_raw, all_predictions_raw, average='weighted', zero_division=0)
recall_raw = recall_score(y_test_raw, all_predictions_raw, average='weighted', zero_division=0)
f1_raw = f1_score(y_test_raw, all_predictions_raw, average='weighted', zero_division=0)

print(f"\nKết quả test trên RAW dataset:")
print(f"Accuracy: {accuracy_raw:.4f}")
print(f"Precision: {precision_raw:.4f}")
print(f"Recall: {recall_raw:.4f}")
print(f"F1-Score: {f1_raw:.4f}")

print(f"\nSo sánh Processed vs Raw:")
print(f"  Processed F1: {f1:.4f}")
print(f"  Raw F1: {f1_raw:.4f}")
print(f"  Difference: {abs(f1 - f1_raw):.4f}")


BƯỚC 7: TEST TRÊN RAW DATASET
----------------------------------------
Raw test data: (175341, 45)
Features after encoding and scaling: (175341, 42)

Kết quả test trên RAW dataset:
Accuracy: 0.9608
Precision: 0.9607
Recall: 0.9608
F1-Score: 0.9605

So sánh Processed vs Raw:
  Processed F1: 0.9523
  Raw F1: 0.9605
  Difference: 0.0083


In [9]:
print("\n" + "=" * 60)
print("HOÀN THÀNH PHASE 4 NETWORK!")
print("=" * 60)
print(f"F1 (Processed): {f1:.4f}")
print(f"F1 (Raw): {f1_raw:.4f}")


HOÀN THÀNH PHASE 4 NETWORK!
F1 (Processed): 0.9523
F1 (Raw): 0.9605
